In [31]:
Month_ = 3
Year_ = 2569
User_ = ["All"]     #Run ข้อมูลของใคร #pao

web_cpiG = 0        #Load ข้อมูลชุดทั่วไป
detectG = 1         #Run การตรวจข้อมูลทั่วไป

web_cpiU = 0        #Load ข้อมูลชุดนอกเขต
detectU = 0         #Run การตรวจข้อมูลนอกเขต

realTimeData = 1    #Load   ไฟล์ตั้งต้นทั้งหมด
standard_price = 0  #Run    ราคากลาง
shopBKK = 0         #Run    ร้านค้า
time_serieG = 0     #Run    timeseries
excess_lostG = 0    #Run    ขาดเกิน

upload = 0          #Update ข้อมูลใหม่ให้เว็บไซต์

In [32]:
1

1

### Condition Input

In [33]:
if detectG == 1:
     sessions = 'g'
elif detectU == 1:
     sessions = 'u'

In [34]:
# pip install --upgrade jupyter ipywidgets

###  Function Section

In [35]:
# from flask import session
# import papermill as pm
# import glob
# import os
# import sys

# # การตั้งค่าการรัน
# KERNEL_NAME = 'python3'

# def run_notebooks(input_path, specific_files=None, **params):

#     current_script = os.path.basename(sys.argv[0])

#     # จัดการประเภทของ Input Path
#     if isinstance(input_path, list):
#         folders_to_process = input_path
#     else:
#         folders_to_process = [input_path]

#     for folder in folders_to_process:
#         if not os.path.exists(folder):
#             print(f"🚫 ไม่พบโฟลเดอร์: {folder}")
#             continue
            
#         abs_folder_path = os.path.abspath(folder)
        
#         # คัดเลือกไฟล์ที่จะรัน
#         if specific_files:
#             if isinstance(specific_files, str):
#                 specific_files = [specific_files]
#             files = [os.path.join(abs_folder_path, f) for f in specific_files]
#         else:
#             all_files = glob.glob(os.path.join(abs_folder_path, "*.ipynb"))
#             files = sorted([f for f in all_files if os.path.basename(f) != current_script])

#         if not files:
#             print(f"⚠️ ไม่พบไฟล์ .ipynb ใน: {folder}")
#             continue

#         print(f"\n📂 Entering Folder: {folder}")
#         if params:
#             print(f"📊 Sending Parameters: {params}")

#         for i, nb in enumerate(files, 1):
#             name = os.path.basename(nb)
#             if not os.path.exists(nb):
#                 print(f"❌ [{i}/{len(files)}] File not found: {name}")
#                 continue

#             print(f"🚀 [{i}/{len(files)}] Running: {name}")
#             try:
#                 # รัน Notebook โดยส่งค่า params เข้าไป
#                 pm.execute_notebook(
#                     nb, 
#                     nb, 
#                     kernel_name=KERNEL_NAME,
#                     parameters=params, # ส่งค่า Month, Year ฯลฯ เข้าไปที่นี่
#                     log_output=False,
#                     cwd=abs_folder_path 
#                 )
#                 print(f"✅ {name}: Success")
#             except Exception as e:
#                 print(f"❌ {name}: Failed! (Skipping...)")
#                 # print(f"Error Detail: {e}") # เปิดบรรทัดนี้เพื่อดูสาเหตุการ Error
                
#                 continue
#     print('-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------')
    
# # print("\n🏁 All processes finished.")

In [36]:
import os, sys, glob, io, contextlib, re
import papermill as pm
from tqdm.notebook import tqdm

KERNEL_NAME = 'python3'

def run_notebooks(input_path, specific_files=None, **params):
    current_script = os.path.basename(sys.argv[0])
    folders = input_path if isinstance(input_path, list) else [input_path]
    
    tasks = []
    for folder in folders:
        if not os.path.exists(folder): continue
        abs_path = os.path.abspath(folder)
        if specific_files:
            files = [os.path.join(abs_path, f) for f in (specific_files if isinstance(specific_files, list) else [specific_files])]
        else:
            # ตัด .ipynb_checkpoints ออกเพื่อความเร็วและแม่นยำ
            files = sorted([f for f in glob.glob(os.path.join(abs_path, "*.ipynb")) 
                           if os.path.basename(f) != current_script and ".ipynb_checkpoints" not in f])
        for f in files: tasks.append((f, abs_path))

    if not tasks: return

    # ปรับ Format: แสดงผลเวลาที่ใช้ไป [elapsed] แทน rate/file
    custom_format = '{bar}| {n_fmt}/{total_fmt} [{elapsed} {postfix}]'

    # ลบ unit="file" ออกเพื่อให้แสดงผลเป็นเวลาดิบๆ (เช่น 3.38s)
    with tqdm(tasks, bar_format=custom_format, leave=True) as pbar:
        for nb_path, abs_path in pbar:
            name_no_ext = os.path.splitext(os.path.basename(nb_path))[0]
            
            f_err = io.StringIO()
            with contextlib.redirect_stderr(f_err):
                try:
                    pm.execute_notebook(
                        nb_path, nb_path, 
                        kernel_name=KERNEL_NAME,
                        parameters=params,
                        log_output=False,
                        progress_bar=False, 
                        cwd=abs_path 
                    )
                except Exception:
                    pbar.write(f"❌ Error: {name_no_ext}")

            # สกัด Unknown Parameters
            error_msg = f_err.getvalue()
            unknown_vars = set(re.findall(r"Passed unknown parameter: (\w+)", error_msg))
            
            # ทำ List Comprehension เพื่อความไว
            param_display = "{" + ", ".join([f"{'' if k in unknown_vars else ''}{k}:{v}" for k, v in params.items()]) + "}"
            
            # อัปเดต Bar แบบประหยัดพื้นที่
            pbar.set_postfix_str(f"ไฟล์: {name_no_ext} {param_display}")


### Running Processes Section

In [ ]:
if __name__ == "__main__":
    # run_notebooks("01-database")
    if realTimeData == 1:
        print("\n Input โดยโหลดข้อมูลตั้งต้น Master/Supplementaryfilre/ราคากลางทุกชุด")
        run_notebooks("01-database", specific_files=["Master.ipynb"], Month=Month_, Year=Year_)
        run_notebooks("01-database", specific_files=["Supplementaryfilre.ipynb"], Month=Month_, Year=Year_)
        # run_notebooks("01-database", specific_files=["StandardPriceAutomation.ipynb"], update_sort_set=False, download_set=True)


    if sessions == "g":
        print("\n Data Analysis คำนวณชุดทั่วไป")
        if shopBKK == 1:
            run_notebooks("01-database", specific_files=["ShopBKK.ipynb"])  
        if web_cpiG == 1: 
            run_notebooks("01-database", specific_files=["data01_cpig_month.ipynb"], Month=Month_, Year=Year_)    
            # run_notebooks("01-database", specific_files=["data02_cpig_week.ipynb"], Month=Month_, Year=Year_)
        if time_serieG == 1:
            run_notebooks("02-timeseries_data", specific_files=["Timeseries_ชุดทั่วไป_เดือน.ipynb"], Month=Month_, Year=Year_)        
        if excess_lostG == 1:
            run_notebooks("04-Excess_Lost", specific_files=["Excess_Lost.ipynb"], Month=Month_, Year=Year_, session = 'g')
        # if center_price ==1:
        if standard_price == 1:
            run_notebooks("01-database", specific_files=["ราคากลาง_concat.ipynb"], Month=Month_, Year=Year_)
        if detectG == 1:
            run_notebooks("03-Detection-G", specific_files=["ตรวจราคาG.ipynb"], Month=Month_, Year=Year_, session = 'g', User = User_)

    if sessions == "u":
        print("Running Supply = u")
        # run_notebooks("01-database", specific_files=["ShopBKK.ipynb"]) 
        # run_notebooks("02-timeseries_data", specific_files=["Timeseries_ชุดนอกเขต_เดือน.ipynb"], Month=Month_, Year=Year_)  
        # run_notebooks("01-database", specific_files=["data03_cpiu_month.ipynb"], Month=Month_, Year=Year_)
        # run_notebooks("01-database", specific_files=["data04_cpiu_week.ipynb"], Month=Month_, Year=Year_)

    if upload == 1:
        run_notebooks("Final_output", specific_files=["CallOutput.ipynb"], Month=Month_, Year=Year_, session = 'g')
    
# ตรวจราคา รัน run_cpi_month/ ราคากลาง บ บ1 น / time_serie

# รายสัปดาห์




 Input โดยโหลดข้อมูลตั้งต้น Master/Supplementaryfilre/ราคากลางทุกชุด


NameError: name 'ห' is not defined

In [40]:
# run_notebooks("01-database", specific_files=["Master.ipynb"], Month=Month_, Year=Year_)

In [17]:
import os, sys, glob, io, contextlib, re, time
import papermill as pm
import pandas as pd
import nbformat
from tqdm.notebook import tqdm
from IPython.display import display

# --- 1. Configuration Section ---
Month_ = 3
Year_ = 2569
User_ = ["All"]

web_cpiG = 0
detectG = 1
web_cpiU = 0
detectU = 0
realTimeData = 1
standard_price = 0
shopBKK = 0
time_serieG = 0
excess_lostG = 0
upload = 0

if detectG == 1:
    sessions = 'g'
elif detectU == 1:
    sessions = 'u'
else:
    sessions = None

KERNEL_NAME = 'python3'

# --- 2. Helper Functions ---

def get_nb_progress(nb_path):
    try:
        with open(nb_path, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
        code_cells = [c for c in nb.cells if c.cell_type == 'code']
        total = len(code_cells)
        executed = len([c for c in code_cells if c.get('execution_count') is not None])
        return round((executed / total) * 100, 1) if total > 0 else 0.0
    except Exception:
        return 0.0

def run_notebooks(input_path, specific_files=None, **params):
    current_script = os.path.basename(sys.argv[0])
    folders = input_path if isinstance(input_path, list) else [input_path]
    
    tasks = []
    for folder in folders:
        if not os.path.exists(folder): continue
        abs_path = os.path.abspath(folder)
        if specific_files:
            files = [os.path.join(abs_path, f) for f in (specific_files if isinstance(specific_files, list) else [specific_files])]
        else:
            files = sorted([f for f in glob.glob(os.path.join(abs_path, "*.ipynb")) 
                           if os.path.basename(f) != current_script and ".ipynb_checkpoints" not in f])
        for f in files: tasks.append((f, abs_path))

    if not tasks: return []

    results = []
    # ปรับแต่ง format ให้แสดง [เวลา , ไฟล์: ชื่อ {params}]
    custom_format = '{percentage:3.0f}%|{bar}| [{elapsed} , ไฟล์: {postfix}]'

    with tqdm(total=100, bar_format=custom_format, leave=True) as pbar:
        for nb_path, abs_path in tasks:
            name_no_ext = os.path.splitext(os.path.basename(nb_path))[0]
            
            # สร้างข้อความแสดงพารามิเตอร์ (กรองเอาเฉพาะตัวแปรสำคัญมาโชว์เพื่อความสะอาด)
            display_params = {k: v for k, v in params.items() if k in ['Month', 'Year', 'session']}
            param_str = f"{name_no_ext} {display_params}"
            pbar.set_postfix_str(param_str)
            
            pbar.reset(total=100) 
            start_time = time.time()
            success = False
            
            f_err = io.StringIO()
            with contextlib.redirect_stderr(f_err):
                try:
                    pm.execute_notebook(
                        nb_path, nb_path, 
                        kernel_name=KERNEL_NAME,
                        parameters=params,
                        log_output=False,
                        progress_bar=False, 
                        cwd=abs_path 
                    )
                    success = True
                    pbar.n = 100 
                except Exception:
                    pbar.n = get_nb_progress(nb_path) 

            pbar.refresh() 
            duration = time.time() - start_time
            
            results.append({
                "ชื่อไฟล์": name_no_ext,
                "ตัวแปรที่ใช้": str(display_params) if display_params else "-", 
                "เวลา (วินาที)": round(duration, 2),
                "รันไปได้ (%)": pbar.n,
                "สถานะ": "✅" if success else "❌"
            })
    return results

# --- 3. Main Execution Section ---
if __name__ == "__main__":
    all_reports = []
    start_all = time.time()

# --- Section: Input Database ---
    if realTimeData == 1:
        print("\n📥 Input: โหลดข้อมูลตั้งต้น Master/Supplementary/ราคากลาง")
        all_reports.extend(run_notebooks("01-database", specific_files=["Master.ipynb"], Month=Month_, Year=Year_))
        all_reports.extend(run_notebooks("01-database", specific_files=["Supplementaryfilre.ipynb"], Month=Month_, Year=Year_))
        # all_reports.extend(run_notebooks("01-database", specific_files=["StandardPriceAutomation.ipynb"], update_sort_set=False, download_set=True))

    # --- Section: Data Analysis (Session G) ---
    if sessions == "g":
        print("\n📊 Data Analysis: คำนวณชุดทั่วไป (G)")
        if shopBKK == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["ShopBKK.ipynb"]))
        
        if web_cpiG == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["data01_cpig_month.ipynb"], Month=Month_, Year=Year_))
            # all_reports.extend(run_notebooks("01-database", specific_files=["data02_cpig_week.ipynb"], Month=Month_, Year=Year_))
        
        if time_serieG == 1:
            all_reports.extend(run_notebooks("02-timeseries_data", specific_files=["Timeseries_ชุดทั่วไป_เดือน.ipynb"], Month=Month_, Year=Year_))
        
        if excess_lostG == 1:
            all_reports.extend(run_notebooks("04-Excess_Lost", specific_files=["Excess_Lost.ipynb"], Month=Month_, Year=Year_, session='g'))
        
        if standard_price == 1:
            all_reports.extend(run_notebooks("01-database", specific_files=["ราคากลาง_concat.ipynb"], Month=Month_, Year=Year_))
            
        if detectG == 1:
            all_reports.extend(run_notebooks("03-Detection-G", specific_files=["ตรวจราคาG.ipynb"], Month=Month_, Year=Year_, session='g', User=User_))

    # --- Section: Supply Analysis (Session U) ---
    if sessions == "u":
        print("\n📦 Running Supply: ชุดนอกเขต (U)")
        # ตัวอย่างการเปิดใช้งานสำหรับ Session U (Uncomment หากต้องการใช้)
        # all_reports.extend(run_notebooks("01-database", specific_files=["ShopBKK.ipynb"]))
        # all_reports.extend(run_notebooks("02-timeseries_data", specific_files=["Timeseries_ชุดนอกเขต_เดือน.ipynb"], Month=Month_, Year=Year_))
        # all_reports.extend(run_notebooks("01-database", specific_files=["data03_cpiu_month.ipynb"], Month=Month_, Year=Year_))
        
    if all_reports:
        print("\n" + "="*85)
        print("📊 ตารางสรุปผลการรัน Notebooks ทั้งหมด")
        print("="*85)
        
        df = pd.DataFrame(all_reports)
        df.index = df.index + 1
        df.index.name = 'ลำดับ'
        
        total_files = len(df)
        success_count = len(df[df['สถานะ'] == "✅"])
        failure_count = total_files - success_count
        
        success_rate = (success_count / total_files) * 100
        total_duration = time.time() - start_all
        
        display(df[["ชื่อไฟล์", "ตัวแปรที่ใช้", "เวลา (วินาที)", "รันไปได้ (%)", "สถานะ"]])
        
        print("-" * 85)
        print(f"⏱️ เวลารวมทั้งสิ้น: {total_duration:.2f} วินาที")
        print(f"✅ สำเร็จ: {success_count}/{total_files} ({success_rate:.1f}%) | ❌ ล้มเหลว: {failure_count} ไฟล์")
        print("="*85)


📥 Input: โหลดข้อมูลตั้งต้น Master/Supplementary/ราคากลาง

📊 Data Analysis: คำนวณชุดทั่วไป (G)
